In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import PanDBIO
from gate import IOStore
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
ios   = IOStore()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource']


# DB-Specific

In [3]:
from lib import metalarchives
mio   = metalarchives.MusicDBIO(verbose=False, mkDirs=False)
apiio = metalarchives.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant MetalArchives DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/MetalArchives


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Master Search:       {0}".format(len(localArtists.get())))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

MetalArchives Search Results
   Local Master Search:       71787
   Global Master Search:      44226
   Global Master Search Data: 0
   Search Artists:            44226
   Errors:                    1177
   Known Summary IDs:         71786


# Search For New Artists

In [6]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

In [8]:
useStarterData  = False
useMasterData   = False
useRYMData      = True
useSpotifyData  = False
useAllMusicData = False # Last one done
useLastFMData   = False
useAOTYData     = False

if useStarterData is True:
    starterData = io.get("starter.p")
    artistNames = Series({v["Ref"].split("/")[-1]: v["Name"] for k,v in starterData.items()})
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useSpotifyData is True:
    spotio = ios.get(db="Spotify")
    spotGenreData = DataFrame(spotio.data.getSummaryNameData()).join(spotio.data.getSummaryGenreData())
    def isMetal(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom", "Hard"]
            for test in tests:
                for genre in genres:
                    if genre.title().find(test) != -1:
                        return True
            return False

    spotGenreData = spotGenreData[spotGenreData["Genre"].notna()]
    artistNames = spotGenreData[spotGenreData["Genre"].apply(isMetal)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]   

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet))) 
elif useAllMusicData is True:
    from gate import MusicDBGate
    mdbgate = MusicDBGate()
    amio = mdbgate.getIO(db="AllMusic")
    amGenreData = DataFrame(amio.data.getSummaryNameData()).join(amio.data.getSummaryGenreData())
    def isMetal(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom", "Hard"]
            for test in tests:
                for genre in genres:
                    if genre.title().find(test) != -1:
                        return True
            return False

    amGenreData = amGenreData[amGenreData["Tag"].notna()]
    artistNames = amGenreData[amGenreData["Tag"].apply(isMetal)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]   

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet))) 
elif useRYMData is True:
    rymio = ios.get(db="RateYourMusic")
    rymGenreData = DataFrame(rymio.data.getSummaryNameData()).join(rymio.data.getSummaryGenreData())
    def isMetal(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom", "core", "Speed", "Tech", "tech"]
            for test in tests:
                for genre in genres:
                    if genre.find(test) != -1:
                        return True
            return False

    rymGenreData = rymGenreData[rymGenreData["Genre"].notna()]
    artistNames = rymGenreData[rymGenreData["Genre"].apply(isMetal)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import MusicDBIO
    pdbio = MusicDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    

#   Artist Names To Get:  8443
#   Artist Names To Get:  4486

MetalArchives Search Results
   Available Names:      6117
   Known Artist Names:   44226
   Artist Names To Get:  1617


## Download Artist Searches

In [9]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "10:30pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchResults(artistName=artistName)
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        apiio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    apiio.sleep(3.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

if True:
    ts.stop()
    print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
    masterArtists.save(data=masterArtistsDict)
    masterArtistsData.save(data=masterArtistsDataDict)
    if len(searchedForErrors) > 0:
        errors.save(data=searchedForErrors)

Process [Getting MetalArchives ArtistIDs] Start    ==> Time Is 2022-04-30 21:30:55
========================= termTime(day=today,time=10:30pm) =========================
   ====> Terminate Time Set To 2022-04-30 22:30:00 <====
   ====> Will Terminate Process 59 Minutes and 5 Seconds From Now
Searching For Jerry Goldsmith                                   0
Searching For Swan Meat                                         0
Searching For CH3                                               0
Searching For Cat's Eyes                                        0
Searching For Daniel Pemberton                                  0
Searching For Placebo                                           1
Searching For Pretty Girls Make Graves                          0
Searching For The Life and Times                                0
Searching For François Tétaz                                  0
Searching For Michael Kamen                                     0
Searching For Richard Harvey                     

Searching For Heavy Pettin                                      1
Searching For Sofiane                                           0
Searching For Daniel Son                                        0
Searching For Danny Brown                                       0
Searching For Kevin Kiner                                       0
Searching For King Krule                                        0
200/?      : Process [Getting MetalArchives ArtistIDs] Has Run For 20 Minutes and 46 Seconds.
Saving 44426 (New=200) MetalArchives Searched For Artist (Info) IDs
Searching For F-777                                             0
Searching For Fashawn                                           0
Searching For Felt                                              0
Searching For Giants Chair                                      0
Searching For Murs                                              0
Searching For Pete & Bas                                        0
Searching For Shredders                       

Searching For Aaron Zigman                                      0
Searching For B.G.'z                                            0
Searching For Boondox                                           0
Searching For Hyph11e                                           0
Searching For No Use for a Name                                 0
Searching For Climax Golden Twins                               0
Searching For Didjits                                           0
Searching For Drugstore                                         0
Searching For Luis Enríquez Bacalov                            0
Searching For Papa M                                            0
Searching For Portobello Bones                                  0
Searching For Yassin                                            0
Searching For From Safety to Where                              0
Searching For Randy Newman                                      0
400/?      : Process [Getting MetalArchives ArtistIDs] Has Run For 41 Minute

## Save Results

In [10]:
from pandas import concat
df = masterArtistsData.get()
if isinstance(df,dict):
    print("Found {0} New Artists".format(len(df)))
    searchDF = metalarchives.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, Series(df)])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    artists = {}
    for artistName,artistResults in searchDF.iteritems():
        if artistResults is not None:
            for item in artistResults:
                artists[item['id']] = item['name']
    print("Found {0} Unique Artists".format(len(artists)))
    s = Series(artists)
    print("  ==> {0} Old Artists".format(len(s[s.index.isin(knownArtists.index)])))
    print("  ==> {0} New Artists".format(len(s[~s.index.isin(knownArtists.index)])))
    print("Saving Data")
    metalarchives.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

Found 575 New Artists
Found 44226 Previous Artists
Found 44801 Total Results
Found 44801 Unique Results
Found 73073 Unique Artists
  ==> 71785 Old Artists
  ==> 1288 New Artists
Saving Data


In [11]:
masterArtistsData.save(data={})

# Download Artist Data

In [12]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

## Find Artists To Download

In [13]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
artistIDsToGet   = artistIDsToGet[~artistIDsToGet.index.isin(errors.get().keys())]

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

#   Artist IDs To Get:  4619

MetalArchives Search Results
   Available IDs:      72986
   Known Artist IDs:   71787
   Artist IDs To Get:  36


## Download The Data

In [14]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
tt = TermTime("tomorrow", "10:50am")
#tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(5.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

Process [Getting MetalArchives Artist Data] Start    ==> Time Is 2022-04-30 22:33:33
========================= termTime(day=tomorrow,time=10:50am) =========================
   ====> Terminate Time Set To 2022-05-01 10:50:00 <====
   ====> Will Terminate Process 12 Hours and 16 Minutes From Now
Searching For Covenant (bands/_/117549)                         True
Searching For Gold für Eisen (bands/_/11807)                    True
Searching For Belzebul (bands/_/14494)                          True
Searching For Acrophet (bands/_/1716)                           True
Searching For Reprobation (bands/_/214)                         True
5/?        : Process [Getting MetalArchives Artist Data] Has Run For 58 Seconds.
Saving 71792 MetalArchives Searched For Artist (Info) IDs
Searching For Obstruction (bands/_/23202)                       True
Searching For Hass (bands/_/26278)                              True
Searching For Disinter (bands/_/3374)                           True
Searching For 

In [ ]:
del searchedForErrors['118073']
errors.save(data=searchedForErrors)


In [17]:
from lib import metalarchives
metalarchives.moveLocalFiles()

from utils import PoolIO
pio = PoolIO("MetalArchives")
pio.parse()
pio.meta()
pio.sum()
pio.search()

MusicDBBaseDirs(db=MetalArchives)
   Using Local Path For Raw Data <<====
   RawDataDir     = /Users/tgadfort/Music/Discog/artists-metalarchives
   ModValDataDir  = /Volumes/Seagate/Discog/artists-metalarchives-db
   MetaDataDir    = /Volumes/Seagate/Discog/artists-metalarchives-db/metadata
   SummaryDataDir = /Users/tgadfort/Music/Discog/db-metalarchives
MetalArchives SummaryData
  ==> Basic
  ==> Media
  ==> Genre
  ==> Bio
  ==> Link
  ==> Metric
ParseRawDataUtils(mdbdata, mdbdir) [MetalArchives]
MetalArchives ModValMetaData
  ==> Basic (Universal)
  ==> Media (Media)
  ==> Genre
  ==> Bio
  ==> Link
MusicDBBaseDirs(db=MetalArchives)
   RawDataDir     = /Volumes/Piggy/Discog/artists-metalarchives
   ModValDataDir  = /Volumes/Seagate/Discog/artists-metalarchives-db
   MetaDataDir    = /Volumes/Seagate/Discog/artists-metalarchives-db/metadata
   SummaryDataDir = /Users/tgadfort/Music/Discog/db-metalarchives
MetalArchives SummaryData
  ==> Basic
  ==> Media
  ==> Genre
  ==> Bio
  ==> 

100%|█████████████████████████████████████████| 100/100 [01:45<00:00,  1.06s/it]

Process [Running imap multiprocessing for 100 mod values ...] Ran For 1 Minute and 46 Seconds    ==> Time Is 2022-04-30 22:57:47
poolMetaIO(numProcs=3)
  ==> MakeFunction: make
  ==> ModVals:      [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
Process [Running imap multiprocessing for 100 mod values ...] Start    ==> Time Is 2022-04-30 22:57:47



100%|█████████████████████████████████████████| 100/100 [00:31<00:00,  3.19it/s]


Process [Running imap multiprocessing for 100 mod values ...] Ran For 31 Seconds    ==> Time Is 2022-04-30 22:58:18
  ====> Saving [71819] ID => Name Basic Summary Data
  ====> Saving [71819] ID => Ref Basic Summary Data
  ====> Saving [71819] ID => Num Albums Basic Summary Data
  ====> Saving [71819] ID => Media Counts Counts Summary Data
  ====> Saving [71819] ID => Album Media Summary Data
  ====> Saving [71819] ID => SingleEP Media Summary Data
  ====> Not Saving ID => Appearance Media Summary Data
  ====> Not Saving ID => Technical Media Summary Data
  ====> Not Saving ID => Mix Media Summary Data
  ====> Saving [71819] ID => Bootleg Media Summary Data
  ====> Not Saving ID => AltMedia Media Summary Data
  ====> Saving [71819] ID => Other Media Summary Data
  ====> Saving [71819] ID => Genre Summary Data
  ====> Saving [71819] ID => Bio Summary Data
  ====> Saving [71819] ID => Link Summary Data
